# Elevate Puan: Simulasi Pipeline Recommendation System

**Studi kasus:** pendamping pertumbuhan bisnis untuk UMKM perempuan  
**Platform:** Google Colab | **Level:** Pemula | **Estimasi:** 60-90 menit

Notebook ini mengikuti pola hands-on instruktur (*Recommendation System dengan Pretrained Model*), tetapi studi kasusnya diganti dengan alur lengkap **Elevate Puan**: dari assessment bisnis sampai growth insight.

## Tujuan pembelajaran
Setelah menyelesaikan notebook ini, kamu dapat:
1. Menjalankan alur Elevate Puan dari awal sampai akhir dengan data contoh (assessment, diagnosis, constraints, finance, rekomendasi, eksperimen, hasil, insight).
2. Membuat **Opportunity Library** (kumpulan playbook) dan mengubahnya menjadi embedding dengan pretrained model.
3. Membangun **recommender content-based** dengan cosine similarity, lalu menambahkan aturan bisnis (filter dan skor gabungan).
4. Membedakan bagian yang dihitung oleh **aturan atau rumus** dan bagian yang memakai **embedding**.
5. Menguji sistem dan menjelaskan keterbatasannya.

> **Catatan penting:** model bahasa tidak otomatis menjadi sistem rekomendasi. Model hanya menghasilkan embedding teks. Logika similarity, aturan bisnis, dan ranking yang kita tulis yang mengubahnya menjadi rekomendasi.

##Gambaran alur sistem

| Tahap di aplikasi | Bagian notebook | Metode |
|---|---|---|
| 1. Business Condition Assessment | Bagian A | Data contoh (persona) |
| 2. Diagnosis Problem Type | Bagian B.1.1 | Diagnosa masalah utama |
| 3. Financial Readiness | Bagian 1.1.2 | Rumus tetap (tanpa AI) |
| 4. Business Reality & Constraints | Bagian B.1.3 | Diagnosa Keterbatasan |
| 5. Playbook strategi | Bagian B.2 | Embedding + cosine similarity |
| 6. Proses rekomendasi | Bagian B.4 | Rule-based Filtering + Weigthed Scoring + Ranking |
| 7. Hasil eksperimen | Bagian C | Penyampaian hasil dengan LLM |

**Istilah penting**
- **Pretrained model:** model yang bobotnya sudah dipelajari dari data besar. Kita tidak melatihnya lagi.
- **Embedding:** representasi numerik dari teks.
- **Cosine similarity:** ukuran kemiripan arah dua vektor. Nilai lebih tinggi berarti lebih mirip menurut model.
- **Content-based recommendation:** merekomendasikan item berdasarkan kemiripan isinya dengan kebutuhan pengguna.
- **Hard filter:** aturan yang menolak item tertentu apa pun skor kemiripannya (misalnya biaya melebihi anggaran).
- **Skor gabungan (hybrid):** gabungan skor kemiripan dan kecocokan jenis masalah.
- **Top-K:** K rekomendasi teratas.

**Batasan notebook ini**
- Ini *content-based recommendation*, bukan collaborative filtering. Sistem tidak belajar dari perilaku pengguna lain.
- Isi playbook di bawah adalah **draf tim**: belum diverifikasi, dan angka biaya hanyalah placeholder.
- Hasil eksperimen (check-in, pesanan tambahan) adalah **simulasi acak dengan seed tetap**, bukan data nyata.
- Narasi pada insight memakai template. LLM (misalnya Gemini) bisa ditambahkan di tahap pengembangan.

##Instalasi library dan import
Jalankan sel berikut. Google Colab biasanya sudah menyediakan Python; kita hanya memasang library tambahan.

In [1]:
%pip -q install sentence-transformers scikit-learn pandas numpy

In [2]:
import copy
import math
import random
import textwrap
from datetime import date, timedelta

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)


def rp(x):
    """Format angka menjadi Rupiah. Contoh: 50000 -> Rp50.000"""
    return "Rp" + f"{int(round(x)):,}".replace(",", ".")


print(rp(3500000))

Rp3.500.000


##**A. INPUT DATA**

Sumber data yang digunakan adalah profil pengguna, assessment, data financial, dan constraint.

Pada data profil pengguna dan assessment terdapat 14 pertanyaan berupa pilihan, checkbox, dan isian (text). Berikut detailnya:
- business_category
- business_age
- purchase_methods
- main_customers
- repeat_frequency
- viewer_behavior
- sales_sources
- sales_trends_3m
- order_surge
- operational_obstacles
- biggest_challenge
- acquisition_activities
- story_text
- main_goal

Pada data financial terdapat 5 pertanyaan singkat berupa isian.
- monthly_revenue
- monthly_operating_cost
- household_need
- product_margin_pct
- willingness_to_lose

Pada data constraint terdapat 3 pertanyaan berupa pilihan.
- hours_per_week
- digital_skill
- team_size

###1. Persona dan Business Condition Assessment
Kita memakai tiga persona. Jawaban assessment memakai **kode tetap** yang sama dengan blueprint, bukan teks tampilan.

| Persona | Ciri utama | Yang ingin ditunjukkan |
|---|---|---|
| **Bu Rina** | Kue kering, penjualan dari WhatsApp dan tetangga, baru mulai, keuangan terbatas | Level 1 (hanya eksperimen gratis) lalu level terbuka setelah siklus 1 |
| **Kak Dinda** | Skincare online, sudah punya 2 eksperimen yang untung | Level 3 terbuka, anggaran lebih besar |
| **Bu Sari** | Pesanan banyak dan sering kewalahan, tetapi ingin penjualan naik | **Gerbang kapasitas**: jangan menambah permintaan dulu |

In [3]:
from google.colab import files
import json
import copy
import pandas as pd

# 1. Memunculkan tombol interaktif untuk mengunggah file .json
print("Silakan unggah file 'personas_data.json' Anda:")
uploaded_files = files.upload()

# 2. PERBAIKAN: Mengambil nama file pertama dengan benar dari dictionary hasil upload
nama_file = list(uploaded_files.keys())[0]

# 3. Membaca data bytes dari file yang diunggah, lalu diubah ke dictionary Python
PERSONAS = json.loads(uploaded_files[nama_file].decode("utf-8"))
print(f"\n[SUKSES] File '{nama_file}' berhasil dimuat ke memori.")
print("Daftar Persona Tersedia:", list(PERSONAS.keys()))

# 4. Daftar kunci wajib sesuai form assessment awalmu
REQUIRED_KEYS = [
    "business_category", "business_age", "purchase_methods", "main_customers",
    "repeat_frequency", "viewer_behavior", "sales_sources", "sales_trends_3m",
    "order_surge", "operational_obstacles", "biggest_challenge",
    "acquisition_activities", "story_text", "main_goal"
]

# 5. Fungsi validasi assessment
def validate_assessment(assessment_data):
    missing = [k for k in REQUIRED_KEYS if k not in assessment_data]
    if missing:
        raise ValueError(f"Data assessment belum lengkap, kekurangan field: {missing}")
    return True

# 6. Memilih persona yang ingin diuji (Pilih: "bu_rina", "kak_dinda", atau "bu_sari")
PILIHAN_PERSONA = "kak_dinda"

# 7. Menyalin data persona terpilih menggunakan deepcopy agar kamus utama aman
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

# 8. Menjalankan validasi assessment
validate_assessment(ctx["assessment"])

# 9. Menampilkan informasi ringkas persona yang sedang aktif diuji
print(f"\n--- Menjalankan Pipeline untuk Persona: {ctx['name']} ---")
print("1. Data Assessment berhasil divalidasi.")
print("2. Data Finansial dimuat:", ctx["finance_input"])
print("3. Data Constraints dimuat:", ctx["constraints"])
print("4. Riwayat Eksperimen (History) tercatat:", len(ctx["history"]), "siklus selesai.")

# 10. Menampilkan data assessment dalam bentuk tabel DataFrame
display(pd.Series(ctx["assessment"], name="Jawaban_Assessment"))

Silakan unggah file 'personas_data.json' Anda:


Saving personas_data.json to personas_data.json

[SUKSES] File 'personas_data.json' berhasil dimuat ke memori.
Daftar Persona Tersedia: ['bu_rina', 'kak_dinda', 'bu_sari']

--- Menjalankan Pipeline untuk Persona: Kak Dinda (Skincare) ---
1. Data Assessment berhasil divalidasi.
2. Data Finansial dimuat: {'monthly_revenue': 12000000, 'monthly_operating_cost': 9000000, 'household_need': 1500000, 'product_margin_pct': 40, 'willingness_to_lose': 300000}
3. Data Constraints dimuat: {'hours_per_week': '5+ jam/minggu', 'digital_skill': 'sedang', 'team_size': '2 – 5 orang'}
4. Riwayat Eksperimen (History) tercatat: 2 siklus selesai.


,Jawaban_Assessment
business_category,Kecantikan & Personal Care
business_age,1–3 tahun
purchase_methods,[Ready stock]
main_customers,[Pelanggan online]
repeat_frequency,Sering
viewer_behavior,Banyak yang bertanya tapi sedikit yang membeli
sales_sources,"[Instagram, TikTok, Marketplace]"
sales_trends_3m,Naik-turun
order_surge,Masih bisa menangani dengan mudah
operational_obstacles,[Membalas customer]


##**B. PROCESS**

Process stage ini tahapannya sbb:

###1. Diagnose
- Rule-based filtering : pencocokan data user (masalah dan kendala) dengan playbook strategi eksperimen. Data user yang dicocokkan antara lain : hasil assessment, data financial, dan constraint.
- Weighted-scoring : memberikan nilai (bobot) pada masing-masing kandidat tersebut (playbook) untuk menentukan mana yang paling prioritas dan paling relevan.

  Faktor Penentu dalam Scoring:

  Biasanya sistem akan memberikan skor tambahan berdasarkan:

  Seberapa dekat playbook tersebut dengan tujuan utama (main_goal) pengguna
  Tingkat kemudahan eksekusi berdasarkan keterbatasan waktu dan tim (constraints).
  
  Riwayat eksperimen sebelumnya (history), agar sistem tidak merekomendasikan strategi yang gagal atau sudah sering dicoba.

- Hasil Akhir (Top Recommendation):
  
  Setelah diberi skor, sistem tinggal mengurutkannya dari skor tertinggi, lalu mengambil 1 hingga 3 playbook teratas untuk disajikan sebagai rekomendasi utama kepada UMKM (misalnya Bu Rina atau Kak Dinda).

###Pemetaan Kategori Masalah

Proses diagnosis dibagi ke dalam 3 Pilar Utama, yang merangkum semua input di atas menjadi kategori yang bisa dibaca oleh kode Python:

---

###1.1 Kategori Masalah Utama (problem_type)

Kategori ini mendiagnosis apa yang sedang macet di bisnis pengguna.

1.low_awareness ("banyak yang melihat tetapi sedikit yang bertanya"
- Penjelasan: Orang-orang sudah mulai tahu atau melihat produk (ada eksposur), tapi ketertarikan awalnya rendah sehingga mereka hanya lewat saja dan enggan engagement (bertanya/chat).
- Diambil dari assessment: Bagian Customer $\rightarrow$ viewer_behavior
- Pilihan jawaban pemicu: "Banyak yang melihat tapi jarang bertanya"

2.low_conversion ("banyak yang bertanya tetapi sedikit yang membeli")
- Penjelasan: Minat awal sudah ada (banyak yang masuk chat/tanya-tanya), tapi ada hambatan di akhir (seperti harga, ragu kualitas, call to action kurang kuat, atau cara closing yang kurang pas) sehingga batal beli.
- Diambil dari assessment: Bagian Customer & Current Challenge
- Pilihan jawaban pemicu:
- viewer_behavior: "Banyak yang bertanya tapi sedikit yang membeli" (atau "...sebagian membeli")
- biggest_challenge: "Banyak yang melihat tapi sedikit yang membeli"

3.weak_retention ("pelanggan jarang membeli kembali")
- Penjelasan: Bisnis terlalu fokus mencari pembeli baru tapi mengabaikan siklus pembeli lama. Tidak ada program lanjutan atau follow-up yang membuat mereka ingat untuk belanja lagi.
- Diambil dari assessment: Bagian Customer & Current Challenge
- Pilihan jawaban pemicu:
- biggest_challenge: "Pelanggan lama jarang kembali"
- repeat_frequency: "Jarang" atau "Belum punya cukup data."

4.capacity_limit ("kewalahan memenuhi pesanan")
- Penjelasan: Kendala operasional di mana lonjakan order atau proses produksi/pengemasan tidak terkejar oleh tenaga/waktu yang ada (hambatan kapasitas internal).
- Diambil dari assessment: Bagian Operations & Current Challenge
- Pilihan jawaban pemicu:
- order_surge: "Mulai kewalahan" atau "Sering kewalahan"
- biggest_challenge: "Operasional mulai kewalahan"
- operational_obstacles: Terisi kendala seperti "Tenaga", "Waktu", atau "Produksi"

5.narrow_market ("pasar sempit dan bergantung pada satu kanal")
- Penjelasan: Jangkauan pasar terlalu terbatas (misal: hanya teman/keluarga atau satu wilayah kecil) dan terlalu bergantung pada satu platform saja (misal: hanya di WhatsApp saja).
- Diambil dari assessment: Bagian Customer & Assessment - Business profile
- Pilihan jawaban pemicu:
- main_customers: Hanya diisi orang terdekat (misal: "Teman & keluarga" / "Tetangga" saja)
- sales_sources: Hanya mengandalkan 1 kanal saja (misal: hanya "WhatsApp" atau hanya "Instagram")

6.inconsistent_promo ("promosi tidak konsisten")
- Penjelasan: Pemilik usaha sering mood-mood-an dalam membuat konten atau promosi—kadang rajin seminggu, lalu menghilang sebulan.
- Diambil dari assessment: Bagian Current Challenge
- Pilihan jawaban pemicu:
- biggest_challenge: "Sulit menjaga konsistensi promosi"

7.product_stagnation ("produk kurang inovasi atau terasa monoton")
- Penjelasan: Produk yang dijual tidak ada pembaruan, model/rasa itu-itu saja, sehingga pembeli lama bosan dan pembeli baru kurang tergiur.
- Diambil dari assessment: Bagian Current Challenge & Acquisition Activities
- Pilihan jawaban pemicu:
- biggest_challenge: "Penjualan stagnan"
- acquisition_activities: Tidak pernah ada aktivitas pembaruan produk (pilihan "Menambah produk" atau "Mengubah packaging" tidak pernah dipilih padahal bisnis sudah berjalan lama).

8.weak_finance_habits ("pencatatan dan pengelolaan uang belum rapi")
- Penjelasan: Uang hasil usaha tercampur dengan dompet pribadi, tidak tahu persis margin bersih, atau kesulitan mengatur modal kerja.
- Diambil dari assessment: Bagian Current Challenge & Financial
- Pilihan jawaban pemicu:
- biggest_challenge: "Sulit mengatur keuangan"

9.hard_to_acquire ("sulit mendapatkan pelanggan baru / krisis traffic"):
- Ini melengkapi jika orang sama sekali tidak ada yang melihat produk.
- Diambil dari assessment: Bagian Current Challenge & Sales Trends
- Pilihan jawaban pemicu:
- biggest_challenge: "Sulit mendapatkan pelanggan baru"
- sales_trends_3m: "Menurun"

10.general_stagnant ("penjualan stagnan & bingung langkah"):
- Untuk pelaku usaha yang bingung apa yang salah karena semua terasa jalan di tempat.
- Diambil dari assessment: Bagian Current Challenge
- Pilihan jawaban pemicu:
- biggest_challenge: "Bingung memilih strategi promosi" atau "Sulit menentukan langkah berikutnya"

---

In [4]:
def diagnose_problem_type(data_user):
    """
    Mendiagnosis problem_type utama dari pengguna berdasarkan data assessment.
    Mendukung list 8 problem_type utama + tambahan logika pelengkap.
    """
    # Menyimpan daftar problem_type yang terdeteksi (bisa lebih dari satu)
    detected_problems = []

    # Ambil data input (dengan nilai default pengaman jika kunci tidak ada)
    viewer_behavior = data_user.get("viewer_behavior", "")
    biggest_challenge = data_user.get("biggest_challenge", "")
    order_surge = data_user.get("order_surge", "")
    sales_sources = data_user.get("sales_sources", [])
    main_customers = data_user.get("main_customers", [])
    repeat_frequency = data_user.get("repeat_frequency", "")

    # 1. Cek low_awareness ("banyak yang melihat tetapi jarang bertanya")
    if viewer_behavior == "Banyak yang melihat tapi jarang bertanya":
        detected_problems.append("low_awareness")

    # 2. Cek low_conversion ("banyak yang bertanya tetapi sedikit yang membeli")
    if viewer_behavior == "Banyak yang bertanya tapi sebagian membeli" or \
       biggest_challenge == "Banyak yang melihat tapi sedikit yang membeli":
        detected_problems.append("low_conversion")

    # 3. Cek weak_retention ("pelanggan jarang membeli kembali")
    if biggest_challenge == "Pelanggan lama jarang kembali" or \
       repeat_frequency in ["Jarang", "Belum punya cukup data."]:
        detected_problems.append("weak_retention")

    # 4. Cek capacity_limit ("kewalahan memenuhi pesanan")
    if "kewalahan" in str(order_surge).lower() or \
       biggest_challenge == "Operasional mulai kewalahan":
        detected_problems.append("capacity_limit")

    # 5. Cek narrow_market ("pasar sempit dan bergantung pada satu kanal")
    if len(sales_sources) == 1 and sales_sources[0] in ["WhatsApp", "Instagram"]:
        detected_problems.append("narrow_market")
    elif "Teman & keluarga" in main_customers and len(main_customers) <= 2:
        detected_problems.append("narrow_market")

    # 6. Cek inconsistent_promo ("promosi tidak konsisten")
    if biggest_challenge == "Sulit menjaga konsistensi promosi":
        detected_problems.append("inconsistent_promo")

    # 7. Cek weak_finance_habits ("pencatatan dan pengelolaan uang belum rapi")
    if biggest_challenge == "Sulit mengatur keuangan":
        detected_problems.append("weak_finance_habits")

    # 8. Cek product_stagnation / general fallback jika belum terdeteksi spesifik
    if biggest_challenge == "Penjualan stagnan":
        detected_problems.append("product_stagnation") # atau bisa diarahkan ke general

    # Fallback jika sama sekali tidak ada rule yang kena
    if not detected_problems:
        detected_problems.append("general_stagnant")

    return detected_problems


# --- Contoh Pengujian (Simulasi Data User) ---
sample_user = {
    "viewer_behavior": "Banyak yang bertanya tapi sebagian membeli",
    "biggest_challenge": "Banyak yang melihat tapi sedikit yang membeli",
    "order_surge": "Masih bisa menangani dengan mudah",
    "sales_sources": ["WhatsApp"],
    "repeat_frequency": "Jarang"
}

print("Hasil Diagnosis Problem Type:", diagnose_problem_type(sample_user))

Hasil Diagnosis Problem Type: ['low_conversion', 'weak_retention', 'narrow_market']


In [5]:
import copy

# --- FUNGSI PENGUJIAN OTOMATIS BERDASARKAN PERSONA AKTIF ---
def run_diagnosis_pipeline(persona_context):
    """
    Fungsi pembungkus untuk menjalankan diagnosis masalah secara dinamis
    berdasarkan data assessment dari persona yang sedang dipilih.
    """
    # Mengambil bagian assessment dari persona aktif
    assessment_data = persona_context["assessment"]

    # Menjalankan fungsi diagnosis yang sudah dibuat sebelumnya
    hasil_diagnosis = diagnose_problem_type(assessment_data)

    # Menampilkan laporan hasil diagnosis ke layar secara rapi
    print(f"=== LAPORAN DIAGNOSIS MASALAH: {persona_context['name']} ===")
    print("-" * 60)
    print("Cerita / Konteks Usaha :", assessment_data.get("story_text", "-"))
    print("Tantangan Terbesar     :", assessment_data.get("biggest_challenge", "-"))
    print("Perilaku Pembeli       :", assessment_data.get("viewer_behavior", "-"))
    print(f">> Problem Type Terdeteksi : {hasil_diagnosis}")
    print("=" * 60)

# ==========================================
# CARA PAKAI: Uji Coba Berbagai Persona
# ==========================================
# Kamu bisa mengganti teks variabel di bawah dengan: "bu_rina", "kak_dinda", atau "bu_sari"
PILIHAN_PERSONA = "kak_dinda"

# Menyalin data persona secara aman agar data asli tidak berubah
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

# Menjalankan pipeline diagnosis otomatis
run_diagnosis_pipeline(ctx)

=== LAPORAN DIAGNOSIS MASALAH: Kak Dinda (Skincare) ===
------------------------------------------------------------
Cerita / Konteks Usaha : Saya sudah rajin posting dan pernah coba iklan, tapi penjualan segitu-segitu saja. Banyak yang DM tanya harga lalu hilang.
Tantangan Terbesar     : Penjualan stagnan
Perilaku Pembeli       : Banyak yang bertanya tapi sedikit yang membeli
>> Problem Type Terdeteksi : ['product_stagnation']


###1.2 Financial Readiness (rumus tetap, tanpa AI)

Sistem melakukan perhitungan kesiapan dana untuk melakukan eksperimen (menggunakan aturan perhitungan, bukan AI).

Konsepnya **Affordable Loss**: yang dihitung adalah apa yang *mampu* dan *mau* hilang oleh pemilik usaha. Semua angka dihitung oleh kode, bukan AI.

```
laba_usaha   = omzet_bulanan - biaya_operasional_bulanan
sisa_bebas   = laba_usaha - kebutuhan_rumah_tangga
batas_aman   = sisa_bebas x risk_allowance
rekomendasi  = MIN(batas_aman, willingness_to_lose)
```


*risk_allowance bergantung pada riwayat eksperimen:

- 15% : jika belum ada riwayat eksperimen / new user

- 20% : jika 2 eksperimen selesai atau lebih

- 25% : jika 2 eksperimen terakhir untung.

---

Aturan Penentuan cost_level (Menyelaraskan Rumus & Growth Ladder)

1.cost_level = 1 (FIN_ZERO_BUDGET):
- Kondisi: Hasil rekomendasi $\le 0$ (atau belum memenuhi syarat level di atasnya).
- Akses Playbook: Hanya eksperimen gratis (Level 1).

2.cost_level = 2 (FIN_LOW_BUDGET):
- Kondisi: Hasil rekomendasi antara Rp1 s.d. Rp100.000, margin sudah diisi, dan minimal 1 siklus eksperimen selesai.
- Akses Playbook: Eksperimen berbayar skala sangat kecil (Level 2).

3.cost_level = 3 (FIN_HEALTHY_BUDGET):
- Kondisi: Hasil rekomendasi $>$ Rp100.000, dua siklus terakhir tidak rugi, dan kas cadangan cukup.
- Akses Playbook: Boleh mengakses Paid Ads, KOL mikro, dll (Level 3).

In [6]:
# --- 1. FUNGSI BANTU FORMAT RUPIAH ---
def rp(x):
    """Mengubah angka integer menjadi format Rupiah (Contoh: 50000 -> Rp50.000)."""
    return "Rp" + f"{int(round(x)):,}".replace(",", ".")

# --- 2. FUNGSI MENENTUKAN PERSENTASE TOLERANSI RISIKO ---
def risk_allowance_pct(history):
    """Menentukan risk_allowance berdasarkan jumlah riwayat dan hasil profit eksperimen."""
    done = len(history)
    # Jika sudah >= 2 eksperimen selesai dan 2 eksperimen terakhir untung (net_profit > 0)
    if done >= 2 and all(h.get("net_profit", 0) > 0 for h in history[-2:]):
        return 0.25  # 25%
    # Jika sudah >= 2 eksperimen selesai (tetapi tidak semuanya untung)
    if done >= 2:
        return 0.20  # 20%
    # Default untuk pengguna baru (new user / belum ada riwayat)
    return 0.15  # 15%

# --- 3. FUNGSI UTAMA PERHITUNGAN FINANSIAL (AFFORDABLE LOSS) ---
def calc_finance(finance_input, history, override_pct=None):
    """Menghitung laba usaha, sisa bebas, batas aman, rekomendasi anggaran, dan status risiko."""
    # Menghitung laba bersih usaha dari omzet dikurangi biaya operasional
    laba = finance_input["monthly_revenue"] - finance_input["monthly_operating_cost"]

    # Menghitung sisa uang bebas setelah dikurangi kebutuhan rumah tangga
    sisa = laba - finance_input["household_need"]

    # Menentukan persentase risk allowance (bisa di-override jika diperlukan)
    pct = override_pct if override_pct is not None else risk_allowance_pct(history)

    # Menghitung batas aman modal (sisa bebas dikali persentase risiko, minimal 0)
    batas_aman = max(0, sisa) * pct

    # Menentukan rekomendasi anggaran (nilai minimum antara batas aman dan kesiapan rugi user)
    rekomendasi = min(batas_aman, finance_input["willingness_to_lose"])

    # Menentukan status kesehatan finansial
    if sisa <= 0:
        status = "not_recommended"
    elif finance_input["willingness_to_lose"] <= batas_aman:
        status = "within_range"
    else:
        status = "above_recommended"

    return {
        "laba_usaha": laba,
        "sisa_bebas": sisa,
        "risk_allowance_pct": pct,
        "batas_aman": batas_aman,
        "recommended_budget": rekomendasi,
        "status": status,
    }

# --- 4. FUNGSI MENENTUKAN LEVEL GROWTH LADDER ---
def calc_growth_level(fin, finance_result, history, level3_cost=150_000, level3_threshold=100_000):
    """Menentukan level akses playbook (Level 1, 2, atau 3) berdasarkan syarat kelayakan."""
    margin_filled = fin.get("product_margin_pct") is not None
    level = 1  # Default level 1 (hanya eksperimen gratis)

    # Syarat naik ke Level 2
    if margin_filled and len(history) >= 1:
        level = 2
        last2 = history[-2:]
        # Syarat naik ke Level 3
        if (finance_result["batas_aman"] >= level3_threshold
                and len(last2) == 2
                and all(h.get("net_profit", 0) >= 0 for h in last2)):
            level = 3
    return level

# Kamus terjemahan label status untuk laporan
STATUS_LABEL = {
    "within_range": "Aman (hijau)",
    "above_recommended": "Hati-hati (kuning)",
    "not_recommended": "Belum disarankan (merah)",
}

# --- 5. FUNGSI UNTUK MENJALANKAN & MENAMPILKAN LAPORAN FINANSIAL SECARA DINAMIS ---
def run_financial_pipeline(persona_context):
    """Fungsi pembungkus agar bisa dipanggil berkali-kali untuk persona apa saja."""
    fin_data = persona_context["finance_input"]
    hist_data = persona_context["history"]

    # Menjalankan kalkulasi finansial
    hasil_fin = calc_finance(fin_data, hist_data)

    # Menentukan level growth ladder
    level_terbuka = calc_growth_level(fin_data, hasil_fin, hist_data)

    # Menampilkan laporan hasil
    print(f"=== LAPORAN KESEHATAN FINANSIAL: {persona_context['name']} ===")
    print("-" * 55)
    print("Laba usaha           :", rp(hasil_fin["laba_usaha"]))
    print("Sisa uang bebas      :", rp(hasil_fin["sisa_bebas"]))
    print(f"Batas aman ({int(hasil_fin['risk_allowance_pct'] * 100)}%)   :", rp(hasil_fin["batas_aman"]))
    print("Rekomendasi anggaran :", rp(hasil_fin["recommended_budget"]))
    print("Status risiko        :", STATUS_LABEL[hasil_fin["status"]])
    print("Level Growth terbuka : Level", level_terbuka)
    print("=" * 55)

In [7]:
# Ganti nama persona di sini sesuai keinginan ("bu_rina", "kak_dinda", atau "bu_sari")
PILIHAN_PERSONA = "kak_dinda"

# Menyalin data persona aktif
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

# Menjalankan fungsi pipeline finansial secara otomatis
run_financial_pipeline(ctx)

=== LAPORAN KESEHATAN FINANSIAL: Kak Dinda (Skincare) ===
-------------------------------------------------------
Laba usaha           : Rp3.000.000
Sisa uang bebas      : Rp1.500.000
Batas aman (25%)   : Rp375.000
Rekomendasi anggaran : Rp300.000
Status risiko        : Aman (hijau)
Level Growth terbuka : Level 3


In [8]:
def diagnose_financial_category(finance_result, level):
    """Menerjemahkan hasil kalkulasi finansial & level menjadi kategori diagnosis risiko."""
    financial_diagnoses = []

    if finance_result["recommended_budget"] <= 0 or finance_result["status"] == "not_recommended":
        financial_diagnoses.append("FIN_ZERO_BUDGET")
    elif finance_result["recommended_budget"] <= 100_000:
        financial_diagnoses.append("FIN_LOW_BUDGET")
    else:
        financial_diagnoses.append("FIN_HEALTHY_BUDGET")

    if finance_result["status"] == "above_recommended":
        financial_diagnoses.append("FIN_RISK_WARNING")

    financial_diagnoses.append(f"COST_LEVEL_{level}")

    return financial_diagnoses

In [9]:
import copy

# --- FUNGSI PENGUJIAN OTOMATIS FINANSIAL BERDASARKAN PERSONA AKTIF ---
def run_financial_diagnosis_pipeline(persona_context):
    """
    Fungsi pembungkus untuk menjalankan diagnosis kategori finansial secara dinamis
    berdasarkan data finansial dan riwayat dari persona yang sedang dipilih.
    """
    fin_data = persona_context["finance_input"]
    hist_data = persona_context["history"]

    # 1. Jalankan kalkulasi finansial dasar (menggunakan fungsi calc_finance yang sudah dibuat sebelumnya)
    hasil_fin = calc_finance(fin_data, hist_data)

    # 2. Tentukan level growth ladder saat ini
    level_terbuka = calc_growth_level(fin_data, hasil_fin, hist_data)

    # 3. Jalankan fungsi diagnosis finansial
    hasil_diagnosis_fin = diagnose_financial_category(hasil_fin, level_terbuka)

    # 4. Menampilkan laporan hasil diagnosis finansial ke layar
    print(f"=== DIAGNOSIS KATEGORI FINANSIAL: {persona_context['name']} ===")
    print("-" * 60)
    print("Omzet Bulanan      :", rp(fin_data["monthly_revenue"]))
    print("Biaya Operasional  :", rp(fin_data["monthly_operating_cost"]))
    print("Kebutuhan Rumah T  :", rp(fin_data["household_need"]))
    print("Rekomendasi Anggaran:", rp(hasil_fin["recommended_budget"]))
    print("Status Finansial   :", hasil_fin["status"])
    print(f">> Tag Diagnosis Finansial : {hasil_diagnosis_fin}")
    print("=" * 60)

# ==========================================
# CARA PAKAI: Uji Coba Berbagai Persona
# ==========================================
# Ganti nama persona sesuai keinginan: "bu_rina", "kak_dinda", atau "bu_sari"
PILIHAN_PERSONA = "kak_dinda"

# Menyalin data persona secara aman
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

# Menjalankan pipeline diagnosis finansial otomatis
run_financial_diagnosis_pipeline(ctx)

=== DIAGNOSIS KATEGORI FINANSIAL: Kak Dinda (Skincare) ===
------------------------------------------------------------
Omzet Bulanan      : Rp12.000.000
Biaya Operasional  : Rp9.000.000
Kebutuhan Rumah T  : Rp1.500.000
Rekomendasi Anggaran: Rp300.000
Status Finansial   : within_range
>> Tag Diagnosis Finansial : ['FIN_HEALTHY_BUDGET', 'COST_LEVEL_3']


**Pertanyaan refleksi**
1. Mengapa batas aman memakai *sisa uang bebas* dan bukan laba usaha saja?
2. Apa yang terjadi pada rekomendasi anggaran jika kebutuhan rumah tangga naik?
3. Ganti `PERSONA` ke `"kak_dinda"` dan jalankan ulang dari Bagian 5. Mengapa levelnya berbeda?

###1.3 Business Reality & Constraints

Bagian ini menjawab pertanyaan: *apa yang benar-benar bisa dilakukan oleh pemilik bisnis ini?* Datanya dipakai sebagai **hard filter** pada rekomendasi (waktu dan kemampuan digital).

1.CONSTRAINT_TIME (Keterbatasan Waktu Ekstrem)
- Kondisi: hours_per_week < 3 jam atau team_size = one man show dengan kesibukan tinggi.
- Dampak ke Playbook: Saring keluar playbook yang butuh produksi konten rumit atau edit video ribet. Pilih playbook yang low-effort tapi berdampak cepat (misal: Broadcast WhatsApp ke kontak lama).

2.CONSTRAINT_SKILL (Keterampilan Digital Rendah)
- Kondisi: digital_skill = rendah.
- Dampak ke Playbook: Hindari playbook teknis yang rumit (seperti setup Pixel Facebook Ads yang njlimet). Berikan panduan step-by-step yang sangat ramah pemula.

In [10]:
# --- 1. FUNGSI DIAGNOSIS KATEGORI CONSTRAINTS ---
def diagnose_constraints(constraints_data):
    """Mendiagnosis batasan waktu, skill digital, dan kapasitas tim dari persona aktif."""
    constraint_tags = []

    # 1. Evaluasi Waktu Luang (Hours per Week)
    hours = constraints_data.get("hours_per_week", "")
    if "3 – 5 jam" in hours or "rendah" in hours.lower():
        constraint_tags.append("TIME_VERY_LIMITED")  # Sangat terbatas, butuh strategi praktis/cepat
    else:
        constraint_tags.append("TIME_SUFFICIENT")    # Waktu cukup untuk eksplorasi

    # 2. Evaluasi Skill Digital (Digital Skill)
    skill = constraints_data.get("digital_skill", "").lower()
    if skill == "rendah":
        constraint_tags.append("SKILL_BEGINNER")     # Butuh playbook yang step-by-step & sangat mudah
    elif skill == "sedang":
        constraint_tags.append("SKILL_INTERMEDIATE") # Bisa menangani tools digital standar
    else:
        constraint_tags.append("SKILL_ADVANCED")     # Mahir teknis digital

    # 3. Evaluasi Kapasitas Tim (Team Size)
    team = constraints_data.get("team_size", "").lower()
    if "one man show" in team or "1" in team:
        constraint_tags.append("TEAM_SOLO")          # Dikerjakan sendiri (perlu otomatisasi/efisiensi)
    else:
        constraint_tags.append("TEAM_WITH_STAFF")    # Ada tim, bisa delegasi tugas

    return constraint_tags

# --- 2. FUNGSI MENAMPILKAN LAPORAN CONSTRAINTS ---
def run_constraint_pipeline(persona_context):
    """Fungsi pembungkus untuk menguji constraint berdasarkan persona yang dipilih."""
    constraints = persona_context["constraints"]
    diagnosis_tags = diagnose_constraints(constraints)

    print(f"=== ANALISIS CONSTRAINTS & KAPASITAS: {persona_context['name']} ===")
    print("-" * 55)
    print("Waktu Luang per Minggu :", constraints.get("hours_per_week"))
    print("Kemampuan Digital      :", constraints.get("digital_skill"))
    print("Kapasitas Tim          :", constraints.get("team_size"))
    print("Tag Diagnosis Kendala  :", diagnosis_tags)
    print("=" * 55)

In [11]:
# Ganti pilihan persona ("bu_rina", "kak_dinda", atau "bu_sari")
PILIHAN_PERSONA = "kak_dinda"

# Menyalin data persona aktif
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

# Menjalankan pipeline constraint
run_constraint_pipeline(ctx)

=== ANALISIS CONSTRAINTS & KAPASITAS: Kak Dinda (Skincare) ===
-------------------------------------------------------
Waktu Luang per Minggu : 5+ jam/minggu
Kemampuan Digital      : sedang
Kapasitas Tim          : 2 – 5 orang
Tag Diagnosis Kendala  : ['TIME_SUFFICIENT', 'SKILL_INTERMEDIATE', 'TEAM_WITH_STAFF']


###2. Membangun Opportunity Library (kumpulan playbook)

**Playbook** adalah satu kartu berisi satu metode untuk satu jenis masalah, lengkap dengan syarat kapan cocok, biaya, waktu, dan metrik. **Opportunity Library** adalah seluruh koleksinya.

Kolom penting:
- `problem_type`: jenis masalah yang dipecahkan (kode tetap).
- `cost_level` dan `est_cost`: level Growth Ladder (1 = gratis, 2 = di bawah Rp50.000, 3 = berbayar) dan perkiraan biaya.
- `skill_needed`, `hours_per_week`, `est_days`: kebutuhan kemampuan dan waktu.
- `adds_demand`: apakah metode ini menambah permintaan (penting kalau usaha sudah kewalahan).
- `source` dan `verified_by`: jejak sumber. Di notebook ini semuanya masih draf.

> **Peringatan:** isi dan angka di bawah ini placeholder untuk simulasi. Sebelum dipakai di produk, tiap kartu harus diverifikasi dari sumber (misalnya ILO SIYB/IYB) dan wawancara pelaku UMKM.

In [12]:
from google.colab import files
import json

# 1. Memunculkan widget interaktif untuk mengunggah file 'playbooks_data.json'
print("Silakan unggah file 'playbooks_data.json' Anda:")
uploaded_playbook_file = files.upload()

# 2. Mengambil nama file yang diunggah secara otomatis dari dictionary hasil upload
nama_file_pb = list(uploaded_playbook_file.keys())[0]

# 3. Membaca data bytes dari file yang diunggah, lalu mengubahnya menjadi List of Dictionaries Python
PLAYBOOKS = json.loads(uploaded_playbook_file[nama_file_pb].decode("utf-8"))

# 4. Menampilkan informasi sukses dan rekap total playbook yang berhasil dimuat
print(f"\n[SUKSES] File '{nama_file_pb}' berhasil dimuat ke memori.")
print(f"Total Playbook Strategi Tersedia: {len(PLAYBOOKS)} strategi.")

# 5. Menampilkan contoh ringkas 3 playbook pertama untuk memastikan struktur data valid
print("\n--- Contoh 3 Playbook Pertama dalam Database ---")
for pb in PLAYBOOKS[:3]:
    print(f"- [{pb['playbook_id']}] {pb['title']} (Problem: {pb['problem_type']} | Cost Level: {pb['cost_level']})")

# Mengonversi list playbook ke DataFrame Pandas, lalu mengambil 5 data teratas dengan seluruh variabelnya
df_playbooks_tampil = pd.DataFrame(PLAYBOOKS).head(5)

print(f"Total Playbook dalam Database: {len(PLAYBOOKS)}")
print("Menampilkan 5 Playbook pertama dengan seluruh variabelnya:")

# Menampilkan tabel interaktif Pandas secara utuh (seluruh kolom)
display(df_playbooks_tampil)

Silakan unggah file 'playbooks_data.json' Anda:


Saving playbooks_data.json to playbooks_data.json

[SUKSES] File 'playbooks_data.json' berhasil dimuat ke memori.
Total Playbook Strategi Tersedia: 50 strategi.

--- Contoh 3 Playbook Pertama dalam Database ---
- [pb-001] Story WhatsApp Terjadwal 3x Seminggu (Problem: low_awareness | Cost Level: 1)
- [pb-002] Katalog Pesan Otomatis & Call-To-Action WhatsApp Business (Problem: low_conversion | Cost Level: 1)
- [pb-003] Paket Bundling Hemat Akhir Pekan (Problem: stagnant_sales | Cost Level: 1)
Total Playbook dalam Database: 50
Menampilkan 5 Playbook pertama dengan seluruh variabelnya:


,playbook_id,title,problem_type,cost_level,cost_estimated,duration_days,min_hours_per_week,min_digital_skill,suitable_for_solo,description,day_by_day_plan,success_metric,gotchas
0,pb-001,Story WhatsApp Terjadwal 3x Seminggu,low_awareness,1,0,7,3,rendah,True,Banyak kontak WhatsApp tetapi jarang berinteraksi. Memposting jadwal konten rutin di WhatsApp St...,"[Hari 1: Pilih 3 produk terlaris dan siapkan foto/video singkat., Hari 2: Post Story 1 - Tampilk...",Jumlah chat masuk per minggu (Target: +5 chat),Hasil terbatas jika daftar kontak di WhatsApp sangat sedikit.
1,pb-002,Katalog Pesan Otomatis & Call-To-Action WhatsApp Business,low_conversion,1,0,7,3,rendah,True,Mengatasi pembeli yang sering bertanya harga lalu menghilang dengan merapikan katalog dan pesan ...,"[Hari 1: Unduh aplikasi WhatsApp Business dan pindahkan akun., Hari 2: Buat katalog 5 produk uta...",Rasio pembeli dibanding penanya (Target: Naik 15%),Pesan otomatis yang terlalu kaku bisa membuat pembeli merasa tidak ramah.
2,pb-003,Paket Bundling Hemat Akhir Pekan,stagnant_sales,1,0,7,5,rendah,True,Meningkatkan nilai transaksi harian dengan menggabungkan produk favorit dan produk kurang laku d...,"[Hari 1: Pilih 1 produk paling laku dan 1 produk pendamping., Hari 2: Buat nama paket bundling y...",Total unit terjual dan rata-rata nilai transaksi (Basket Size),Pastikan margin profit produk utama tetap terjamin meskipun didiskon.
3,pb-004,Sistem Pre-Order (PO) Terjadwal Batching,operations_overwhelmed,1,0,7,5,rendah,True,Mengatasi kewalahan operasional dengan membatasi hari produksi dan pengiriman dalam sistem klote...,"[Hari 1: Tentukan jadwal pengiriman khusus (misal: Hanya hari Rabu dan Sabtu)., Hari 2: Buat pen...",Ketepatan waktu pengiriman 100% tanpa waktu kerja berlebih,Pelanggan impulsif yang butuh mendadak mungkin akan menolak sistem PO.
4,pb-005,Program Kartu Loyalitas / Diskon Pembelian Ke-5,low_repeat_orders,1,0,7,3,rendah,True,Mendorong pelanggan lama untuk membeli kembali secara berkala menggunakan sistem penandaan trans...,"[Hari 1: Rancang kupon cetak sederhana / kartu cetak stempel pembelian., Hari 2: Tentukan hadiah...",Jumlah repeat order dari pelanggan lama dalam minggu ini,Harus konsisten mencatat riwayat transaksi pelanggan agar tidak keliru.


---

###Memuat pretrained model
Kita memakai model **multilingual** karena playbook dan cerita pengguna berbahasa Indonesia. Contoh hands-on instruktur memakai `all-MiniLM-L6-v2` yang berorientasi bahasa Inggris. Model multilingual ini juga yang dipakai pada notebook RAG kelas.

Unduhan pertama bisa memerlukan waktu dan koneksi internet.

In [13]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)
print("Model berhasil dimuat:", MODEL_NAME)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


### Mengubah playbook menjadi embedding
Teks yang di-embed adalah gabungan judul, jenis masalah, kondisi cocok, dan ringkasan. Kolom lain (biaya, waktu, langkah) tidak di-embed. Kolom-kolom itu dipakai oleh aturan.

Seperti pada hands-on instruktur, embedding katalog dihitung **satu kali**, lalu dipakai berulang kali untuk setiap pengguna.

In [14]:
import pandas as pd

# 1. Pastikan variabel PLAYBOOKS (list of dict dari json) sudah diubah dulu ke DataFrame Pandas
playbooks = pd.DataFrame(PLAYBOOKS)

# 2. Menggabungkan teks penting dari playbook menjadi satu kalimat utuh (disesuaikan dengan kolom di JSON kita)
playbooks["embed_text"] = playbooks.apply(
    lambda r: f"{r['title']}. Kategori Masalah: {r['problem_type']}. "
              f"Deskripsi: {r['description']}",
    axis=1,
)

# 3. Mengubah teks gabungan tersebut menjadi vektor embedding menggunakan model SentenceTransformer
playbook_embeddings = model.encode(
    playbooks["embed_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

# 4. Mencetak informasi bentuk matriks hasil embedding
print("\n[SUKSES] Playbook berhasil diubah menjadi embedding!")
print("Bentuk matriks embedding:", playbook_embeddings.shape)
print("Jumlah playbook:", len(playbooks))
print("Dimensi embedding per playbook:", playbook_embeddings.shape[1])

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[SUKSES] Playbook berhasil diubah menjadi embedding!
Bentuk matriks embedding: (50, 384)
Jumlah playbook: 50
Dimensi embedding per playbook: 384


**Cara membaca output:** jika bentuknya `(50, 384)`, berarti ada 14 playbook dan masing-masing direpresentasikan oleh 384 angka. Angka-angka itu bukan label, melainkan pola semantik yang dipelajari model.

###Embedding Teks Masalah Pengguna (story_text)

Mengubah cerita atau keluhan persona yang sedang aktif (misalnya Bu Rina atau Kak Dinda) menjadi vektor angka agar bisa dibandingkan dengan database playbook.

In [15]:
# 1. Mengambil teks cerita/keluhan dari assessment persona aktif
user_story = ctx["assessment"]["story_text"]
print(f"Cerita/Masalah Persona ({ctx['name']}):\n\"{user_story}\"")

# 2. Mengubah teks masalah pengguna menjadi vektor embedding menggunakan model AI
user_embedding = model.encode([user_story], normalize_embeddings=True)

print("\n[SUKSES] Teks masalah pengguna berhasil di-embedding.")
print("Bentuk vektor user:", user_embedding.shape)

Cerita/Masalah Persona (Kak Dinda (Skincare)):
"Saya sudah rajin posting dan pernah coba iklan, tapi penjualan segitu-segitu saja. Banyak yang DM tanya harga lalu hilang."

[SUKSES] Teks masalah pengguna berhasil di-embedding.
Bentuk vektor user: (1, 384)


###3. Perhitungan Kesamaan Semantik (Cosine Similarity)

Menghitung tingkat kemiripan makna antara masalah pengguna dengan seluruh 50 playbook yang ada di database menggunakan operasi matriks.

In [16]:
import numpy as np

# 1. Menghitung dot product antara vektor user dan matriks embedding playbook (karena sudah dinormalisasi, ini setara dengan Cosine Similarity)
similarities = np.dot(playbook_embeddings, user_embedding.T).flatten()

# 2. Memasukkan skor kesamaan ke dalam DataFrame Pandas
playbooks["similarity_score"] = similarities

# 3. Mengurutkan playbook berdasarkan skor kemiripan tertinggi ke terendah
ranked_playbooks = playbooks.sort_values(by="similarity_score", ascending=False)

print("[SUKSES] Skor kesamaan semantik berhasil dihitung untuk semua playbook.")
print("Top 3 Playbook dengan kecocokan semantik tertinggi:")
display(ranked_playbooks[["playbook_id", "title", "problem_type", "similarity_score"]].head(3))

[SUKSES] Skor kesamaan semantik berhasil dihitung untuk semua playbook.
Top 3 Playbook dengan kecocokan semantik tertinggi:


,playbook_id,title,problem_type,similarity_score
24,pb-025,Iklan Shopee / Tokopedia Ads Kunci Spesifik (Exact Match),stagnant_sales,0.552594
26,pb-027,Flash Sale Terbatas Durasi 3 Jam di Media Sosial,stagnant_sales,0.522547
9,pb-010,Eksperimen Headline Promosi Fokus Solusi,inconsistent_promo,0.510421


###4. Sistem Filtering Berlapis, Scoring & Pemeringkatan Akhir

Menyaring hasil pencarian semantik agar playbook yang direkomendasikan benar-benar selaras dengan diagnosis masalah utama (problem_type) pengguna, aman secara finansial (cost_level sesuai batas aman), serta sesuai dengan kendala waktu dan skill digital yang dimiliki.

In [17]:
import copy
import pandas as pd
import numpy as np

# ==========================================
# 1. PERSIAPAN & PEMANGGILAN PERSONA AKTIF
# ==========================================
PILIHAN_PERSONA = "kak_dinda" # Bisa diganti: "bu_rina", "kak_dinda", "bu_sari"
ctx = copy.deepcopy(PERSONAS[PILIHAN_PERSONA])

print(f"=== MENJALANKAN PIPELINE UNTUK: {ctx['name']} ===")

# ==========================================
# 2. STEP FINANSIAL & LEVEL GROWTH
# ==========================================
hasil_fin = calc_finance(ctx["finance_input"], ctx["history"])
user_max_level = calc_growth_level(ctx["finance_input"], hasil_fin, ctx["history"])

print(f"[Finansial] Status: {hasil_fin['status']} | Max Cost Level yang Diizinkan: Level {user_max_level}")

# ==========================================
# 3. STEP DIAGNOSIS MASALAH UTAMA (PROBLEM TYPE)
# ==========================================
user_detected_problems = diagnose_problem_type(ctx["assessment"])
print(f"[Diagnosis] Problem Type Terdeteksi: {user_detected_problems}")

# ==========================================
# 4. STEP KELOLA CONSTRAINT (WAKTU & SKILL DIGITAL)
# ==========================================
# Konversi skill digital teks menjadi angka hierarki agar bisa difilter secara matematis
skill_mapping = {"rendah": 1, "sedang": 2, "tinggi": 3}
user_skill_text = ctx["constraints"].get("digital_skill", "rendah").lower()
user_skill_level = skill_mapping.get(user_skill_text, 1)

print(f"[Constraints] Kapasitas Waktu: {ctx['constraints'].get('hours_per_week')} | Skill Level: {user_skill_level} ({user_skill_text})")

# ==========================================
# 5. STEP SEMANTIC SEARCH (EMBEDDING & SIMILARITY)
# ==========================================
# Menggunakan teks cerita assessment user untuk dicocokkan dengan playbook
user_story = ctx["assessment"]["story_text"]
user_embedding = model.encode([user_story], normalize_embeddings=True)

# Hitung kesamaan semantik dengan seluruh playbook
similarities = np.dot(playbook_embeddings, user_embedding.T).flatten()
playbooks["similarity_score"] = similarities

# ==========================================
# 6. STEP FILTERING BERLAPIS (KEAMANAN FINANSIAL & KENDALA)
# ==========================================
filtered_playbooks = []
for _, row in playbooks.iterrows():

    # Filter 1: Batas Level Biaya (Affordable Loss)
    if row["cost_level"] > user_max_level:
        continue

    # Filter 2: Batas Skill Digital (Mencegah playbook yang terlalu rumit untuk pemula)
    # Ubah min_digital_skill playbook ke angka
    pb_skill_level = skill_mapping.get(str(row["min_digital_skill"]).lower(), 1)
    if user_skill_level < pb_skill_level:
        continue

    # Filter 3: Batas Waktu / Jam per Minggu
    # Jika playbook membutuhkan jam minimum lebih besar dari kapasitas user (misal user batas 5 jam, playbook butuh 10 jam)
    # Kita bisa beri penalti atau filter tegas. Di sini kita buat sistem filter tegas jika kapasitas terlalu jomplang.
    user_hours_str = str(ctx["constraints"].get("hours_per_week", "")).lower()
    if "3 – 5" in user_hours_str and row["min_hours_per_week"] > 5:
        continue

    filtered_playbooks.append(row.to_dict())

print(f"[Filtering] Total Playbook lolos dari filter finansial & kendala: {len(filtered_playbooks)} dari 50")

# ==========================================
# 7. STEP SCORING (BOBOT KECOCOKAN MASALAH UTAMA)
# ==========================================
scored_playbooks = []
for pb in filtered_playbooks:
    base_score = pb["similarity_score"]

    # Berikan bonus bobot yang kuat (+0.35) jika problem_type playbook
    # benar-benar ada di dalam hasil diagnosis akar masalah user
    if pb["problem_type"] in user_detected_problems:
        problem_match_bonus = 0.35
    else:
        problem_match_bonus = 0.0

    pb["final_score"] = base_score + problem_match_bonus
    scored_playbooks.append(pb)

# ==========================================
# 8. STEP RANKING & OUTPUT TOP 3 REKOMENDASI
# ==========================================
df_ranked = pd.DataFrame(scored_playbooks).sort_values(by="final_score", ascending=False)

print("\n" + "="*65)
print(f"🏆 REKOMENDASI TOP 3 STRATEGI EKSPERIMEN UNTUK: {ctx['name']}")
print("="*65)

for idx, rec in df_ranked.head(3).iterrows():
    print(f"\n🔹 [{rec['playbook_id']}] {rec['title']}")
    print(f"   - Kategori Masalah : {rec['problem_type']}")
    print(f"   - Level Biaya      : Level {rec['cost_level']} (Estimasi: Rp{rec['cost_estimated']:,})")
    print(f"   - Skor Akhir       : {rec['final_score']:.3f} (Semantik: {rec['similarity_score']:.3f})")
    print(f"   - Aksi 7 Hari      :")
    for step in rec["day_by_day_plan"]:
        print(f"     * {step}")

=== MENJALANKAN PIPELINE UNTUK: Kak Dinda (Skincare) ===
[Finansial] Status: within_range | Max Cost Level yang Diizinkan: Level 3
[Diagnosis] Problem Type Terdeteksi: ['product_stagnation']
[Constraints] Kapasitas Waktu: 5+ jam/minggu | Skill Level: 2 (sedang)
[Filtering] Total Playbook lolos dari filter finansial & kendala: 47 dari 50

🏆 REKOMENDASI TOP 3 STRATEGI EKSPERIMEN UNTUK: Kak Dinda (Skincare)

🔹 [pb-027] Flash Sale Terbatas Durasi 3 Jam di Media Sosial
   - Kategori Masalah : stagnant_sales
   - Level Biaya      : Level 1 (Estimasi: Rp0)
   - Skor Akhir       : 0.523 (Semantik: 0.523)
   - Aksi 7 Hari      :
     * Hari 1: Tentukan 1 item stok melimpah yang akan dijadikan produk promo.
     * Hari 2: Buat flyer promo 'Grebek Flash Sale 3 Jam Saja (16.00–19.00)'.
     * Hari 3: Posting pengumuman countdown (H-1) di Story WA dan Instagram.
     * Hari 4: Kirim reminder pesan 1 jam sebelum Flash Sale dimulai.
     * Hari 5: Eksekusi Flash Sale dan posting pembaruan sisa stok y

##**C. OUTPUT**

#Flow LLM Generator

Jadi, sistem rekomendasi deterministik kita (yang berbasis skor, filtering, dan diagnosis) bertindak sebagai otak/analis data yang memastikan akurasi finansial dan kecocokan masalahnya.

Lalu, Gemini AI bertindak sebagai coaching partner yang hangat dan suportif untuk menterjemahkan data dingin tersebut menjadi bahasa manusia yang memotivasi UMKM.

---

Untuk menghubungkannya, berikut step yang dilakukan:
1. Mengambil data teratas dari hasil df_ranked (Playbook peringkat 1).
2. Merangkai variabel (nama user, nama usaha, masalah terdiagnosis, anggaran aman, dan detail playbook 7 hari) ke dalam sebuah Prompt Template.
3. Memanggil API Gemini untuk men-generate copywriting yang natural, empatik, dan humanis.

In [18]:
!pip install google-genai

In [19]:
import os
from getpass import getpass
from google import genai

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY") or getpass("Masukkan GEMINI_API_KEY: ")

client = genai.Client(api_key=api_key)

from google import genai


def generate_coaching_message(ctx, hasil_fin, top_recommendation):
    nama_user = ctx['name']
    nama_usaha = ctx.get('name', 'Usaha Anda')
    cerita_user = ctx['assessment']['story_text']
    budget_aman = rp(hasil_fin['recommended_budget'])

    pb_title = top_recommendation['title']
    pb_problem = top_recommendation['problem_type']
    pb_cost = rp(top_recommendation['cost_estimated'])
    pb_steps = "\n".join([f"- {step}" for step in top_recommendation['day_by_day_plan']])

    prompt = f"""
    Kamu adalah seorang mentor bisnis yang hangat, ramah, sangat suportif, dan tidak menghakimi untuk UMKM di Indonesia.
    Tugasmu adalah menjelaskan hasil rekomendasi strategi eksperimen 7 hari kepada pemilik usaha berdasarkan data di bawah ini.

    DATA PEMILIK USAHA & BISNIS:
    - Nama & Usaha: {nama_user}
    - Cerita/Keluhan: "{cerita_user}"
    - Batas Anggaran Aman: {budget_aman} (Gunakan dana ini sebagai acuan bahwa eksperimen ini aman bagi keuangan mereka).

    REKOMENDASI STRATEGI TERPILIH:
    - Judul Playbook: {pb_title}
    - Kategori Masalah yang Ditangani: {pb_problem}
    - Estimasi Biaya: {pb_cost}
    - Rencana Aksi 7 Hari:
    {pb_steps}

    INSTRUKSI PENULISAN:
    1. Sapa pemilik usaha dengan hangat, validasi perasaannya (misal: akui bahwa tantangan omzet/penjualan itu wajar dan bisa diatasi pelan-pelan).
    2. Jelaskan **mengapa** strategi ini dipilihkan khusus untuk mereka (hubungkan keluhan mereka dengan solusi di playbook ini secara natural).
    3. Yakinkan mereka secara finansial (ingatkan bahwa estimasi biayanya aman dan sesuai dengan batas anggaran mereka, jadi tidak perlu cemas).
    4. Jabarkan langkah 7 hari dengan gaya bahasa yang memotivasi, seperti teman diskusi yang siap mendampingi, bukan instruksi robotik yang kaku.
    5. Gunakan bahasa Indonesia yang santai, akrab, profesional, dan menenangkan.
    """

    # Menggunakan syntax baru dari google-genai
    response = client.models.generate_content(
        model='gemini-3-flash-preview',
        contents=prompt,
    )

    return response.text

In [ ]:
# Inisialisasi client (pastikan API key sudah diatur di Colab Secrets atau env)
# client = genai.Client(api_key="MASUKKAN_API_KEY_KAMU")
# Atau jika menggunakan Colab userdata:
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [20]:
# Ambil playbook peringkat teratas (ranking 1)
top_rec = df_ranked.head(1).iloc[0].to_dict()

# Generate narasi hangat dari Gemini
pesan_coaching = generate_coaching_message(ctx, hasil_fin, top_rec)

# Tampilkan hasilnya ke user
print(pesan_coaching)

Halo, Kak Dinda! Apa kabarnya bisnis skincare-nya hari ini? Semoga tetap semangat ya, Kak.

Aku paham banget rasanya kalau sudah rajin posting, bikin konten, bahkan sampai coba iklan, tapi penjualannya rasanya "segitu-segitu saja". Apalagi kalau ketemu calon pembeli yang cuma DM tanya harga lalu menghilang tanpa jejak—rasanya gemas sekaligus lelah ya, Kak? Tapi Kak Dinda jangan berkecil hati dulu, karena tantangan ini wajar banget dialami banyak pemilik usaha. Hal ini biasanya terjadi karena calon pembeli Kakak sebenarnya tertarik, tapi mereka belum merasa "butuh sekarang juga".

Nah, untuk mengatasi itu, aku sudah siapkan satu strategi khusus buat Kak Dinda yang disebut **"Grebek Flash Sale 3 Jam"**. Strategi ini aku pilihkan supaya orang-orang yang tadinya cuma hobi tanya-tanya, jadi merasa harus segera *checkout* karena takut kehabisan.

Kabar baiknya, strategi ini estimasi biayanya **Rp0 alias gratis!** Mengingat Kak Dinda punya batas anggaran aman Rp300.000, eksperimen ini sangat 